# Residuo sobre baseline: perfeccionar el promedio en vez de competirle

Un notebook que hace todo — preprocesamiento, features, comparación de esquemas, Optuna,
entrenamiento final y submit — para probar una idea concreta.

## La evidencia que motiva el diseño

| Modelo | WAPE observado |
|---|---|
| Promedio de los últimos meses | 0,273 |
| LightGBM | 0,280 |
| **Regresión lineal stepwise** | **0,230** |

Dos lecturas, y las dos apuntan al mismo lado:

**1. El GBM no le gana a un promedio de 3 meses.** Con cientos de features. Eso no dice
que las features sean inútiles: dice que el GBM está gastando su capacidad en aprender
*el nivel* de cada serie, que es la mayor parte de la varianza y que un promedio ya
resuelve.

**2. Una regresión lineal le gana por 18 %.** Y eso es muy informativo: significa que la
señal es aproximadamente una **combinación lineal de los niveles recientes**,
`tn(t+2) ≈ a·tn(t) + b·tn(t−1) + c·tn(t−2) + …`. O sea un promedio móvil con **pesos
aprendidos** en vez de fijos.

Los árboles son estructuralmente malos para eso: aproximan una recta con escalones,
necesitan decenas de cortes para lo que un coeficiente hace solo, y no extrapolan fuera
del rango que vieron.

## Lo que se prueba, entonces

No «residuo sí o no», sino **quién se queda con cada parte del problema**:

| Esquema | Nivel | Desviación |
|---|---|---|
| **A** baseline solo | promedio móvil | — |
| **B** LightGBM al nivel | árbol | árbol |
| **C** lineal al nivel | recta | recta |
| **D** LightGBM al residuo | promedio móvil | árbol |
| **E** lineal + LightGBM al residuo | **recta** | **árbol** |
| **F** LightGBM con hojas lineales | árbol con rectas adentro | |

El **E** es la apuesta: la recta hace lo que hace bien (el nivel, extrapolando), el árbol
hace lo que hace bien (interacciones y quiebres sobre lo que sobra). El **F** es la
versión que LightGBM trae de fábrica con `linear_tree=True`, que ajusta una regresión
lineal *dentro de cada hoja*.

Los seis se miden con **el mismo WAPE en toneladas**, la misma partición y la misma
semilla. La única diferencia es la descomposición.

## Granularidad

El panel es **producto-mes**: es el nivel en el que evalúa Kaggle, y son ~28.000 filas
en vez de 9 millones. Eso importa acá porque el objetivo del notebook es **iterar rápido
sobre la pregunta de modelado** — con este tamaño, los seis esquemas se comparan en
minutos y Optuna corre en una noche sin discusión.

Son **todos** los productos y **todos** los clientes: la dimensión cliente se agrega
sumando, que es exactamente lo que hace la métrica de la competencia antes de medir.

## 0 — Ambiente

In [ ]:
import gc, json, os, shutil, subprocess, time
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.linear_model import Ridge

optuna.logging.set_verbosity(optuna.logging.WARNING)


def resolver_bucket() -> Path:
    env = os.environ.get("LABO3_BUCKET")
    if env and Path(env).expanduser().exists():
        return Path(env).expanduser().resolve()
    for cand in (Path.home() / "buckets" / "b1",
                 "/content/buckets/b1", "/home/ds/buckets/b1"):
        if Path(cand).is_dir():
            return Path(cand)
    raise RuntimeError("No encontre el bucket. Defini LABO3_BUCKET.")


BUCKET   = resolver_bucket()
DIR_RAW  = BUCKET / "datasets"
RUTA_EXP = BUCKET / "exp_residuo"
RUTA_EXP.mkdir(parents=True, exist_ok=True)

print(f"BUCKET: {BUCKET}")
print(f"salida: {RUTA_EXP}")

## 1 — Palancas

In [ ]:
def rango_meses(desde: int, hasta: int) -> list:
    a, b = (desde // 100) * 12 + desde % 100, (hasta // 100) * 12 + hasta % 100
    return [((m - 1) // 12) * 100 + ((m - 1) % 12) + 1 for m in range(a, b + 1)]


PARAM = {
    # ── Datos ────────────────────────────────────────────────────────────
    'solo_productos_target': True,   # los 780 que se entregan
    'muestra_productos': None,       # None = todos. Un numero para probar el pipe rapido.
    'horizonte': 2,
    'max_lags': 12,

    # ── Particion (misma que el pipe, para poder comparar) ───────────────
    'meses_train': rango_meses(201701, 201905),
    'meses_val':   [201907, 201908],
    'meses_test':  [201910],
    'reentrenar_con_val_para_test': True,

    # ── EL BASELINE ──────────────────────────────────────────────────────
    # Sobre que se calcula el residuo. 'auto' prueba todos y elige por VALIDACION.
    #   'tn0'      -> repetir el ultimo mes (el naive)
    #   'ma3/6/12' -> promedio movil de N meses
    #   'ma_pond'  -> 0.5*tn0 + 0.3*tn1 + 0.2*tn2, pesos fijos
    #   'lineal'   -> Ridge sobre los lags: el promedio movil con pesos APRENDIDOS.
    #                 Es la version formal de la regresion stepwise que dio 0,23.
    'baseline': 'auto',

    # ── EL ESQUEMA ───────────────────────────────────────────────────────
    # 'auto' compara los seis y sigue con el mejor en validacion.
    # Forzarlo sirve para aislar un esquema y compararlo en el leaderboard.
    #   'A_baseline' | 'B_lgbm_nivel' | 'C_lineal_nivel'
    #   'D_lgbm_residuo' | 'E_lineal_mas_lgbm' | 'F_lgbm_hojas_lineales'
    'esquema': 'auto',

    # ── Optuna sobre el esquema ganador ──────────────────────────────────
    'n_trials': 40,
    'techo_arboles': 800,

    # ── Ridge ────────────────────────────────────────────────────────────
    # Regularizacion de la parte lineal. Con lags muy correlacionados entre si
    # (que es el caso) sin regularizar los coeficientes se vuelven inestables.
    'ridge_alpha': 1.0,

    # ── Entrega ──────────────────────────────────────────────────────────
    'periodo_objetivo': 202002,
    'semillas_ensemble': [102191],
    'clip_min': 0.0,
    'kaggle_competition': 'labo-iii-2026-rosario',
    'submit': False,
    'mensaje_submit': None,

    'semilla': 102191,
    'sufijo': '',
}

H = PARAM['horizonte']
L = PARAM['max_lags']

EXPERIMENTO = (f"residuo_p_{L}lags_base-{PARAM['baseline']}_esq-{PARAM['esquema']}"
               f"_val{PARAM['meses_val'][0]}-{PARAM['meses_val'][-1]}"
               f"_test{PARAM['meses_test'][0]}"
               + (f"_{PARAM['sufijo']}" if PARAM['sufijo'] else ""))
DIR_OUT = RUTA_EXP / EXPERIMENTO
DIR_OUT.mkdir(parents=True, exist_ok=True)

print(f"EXPERIMENTO: {EXPERIMENTO}")
print(f"carpeta    : {DIR_OUT.relative_to(BUCKET)}")

## 2 — Preprocesamiento y features

Panel producto-mes densificado dentro de la vida de cada producto, con lags, promedios
móviles, shares en los tres niveles de categoría, deltas de share e índices. Todo
causal: cada feature de la fila `t` usa sólo información hasta `t`.

In [ ]:
t0 = time.time()

sell = pl.read_csv(DIR_RAW / "sell-in.txt.gz", separator="\t")
prod = (pl.read_csv(DIR_RAW / "tb_productos.txt", separator="\t")
          .unique(subset=["product_id"]))
target_ids = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")["product_id"].to_list()

if PARAM['solo_productos_target']:
    sell = sell.filter(pl.col("product_id").is_in(target_ids))
if PARAM['muestra_productos']:
    _top = (sell.group_by("product_id").agg(pl.col("tn").sum().alias("t"))
                .sort("t", descending=True).head(PARAM['muestra_productos'])["product_id"])
    sell = sell.filter(pl.col("product_id").is_in(_top.to_list()))

print(f"sell-in: {sell.height:,} filas · {sell['product_id'].n_unique()} productos "
      f"· {sell['customer_id'].n_unique()} clientes")

# ── Panel producto-mes: se colapsa la dimension cliente, que es la misma
#    agregacion que hace la metrica de la competencia antes de medir.
panel = (sell.group_by(["product_id", "periodo"])
             .agg(pl.col("tn").sum().alias("tn"),
                  pl.col("cust_request_tn").sum().alias("req_tn"),
                  pl.col("cust_request_qty").sum().alias("req_qty"),
                  pl.col("customer_id").n_unique().alias("n_clientes"),
                  pl.col("plan_precios_cuidados").max().alias("precios_cuidados"))
             .with_columns((((pl.col("periodo") // 100) * 12)
                            + (pl.col("periodo") % 100)).alias("m")))

vida = panel.group_by("product_id").agg(pl.col("m").min().alias("m_nace"),
                                        pl.col("m").max().alias("m_muere"))
grilla = (vida.with_columns(pl.int_ranges("m_nace", pl.col("m_muere") + 1).alias("m"))
              .explode("m").select("product_id", "m"))

panel = (grilla.join(panel.drop("periodo"), on=["product_id", "m"], how="left")
               .with_columns(pl.col("tn").fill_null(0.0), pl.col("req_tn").fill_null(0.0),
                             pl.col("req_qty").fill_null(0), pl.col("n_clientes").fill_null(0),
                             pl.col("precios_cuidados").fill_null(0))
               .join(vida, on="product_id", how="left")
               .join(prod.select("product_id", "cat1", "cat2", "cat3", "brand", "sku_size"),
                     on="product_id", how="left")
               .with_columns(
                   ((((pl.col("m") - 1) // 12) * 100) + ((pl.col("m") - 1) % 12) + 1)
                     .alias("periodo"),
                   pl.when(pl.col("m") >= pl.col("m_nace"))
                     .then(pl.col("m") - pl.col("m_nace")).otherwise(-1).alias("edad"))
               .sort(["product_id", "m"]))

print(f"panel: {panel.height:,} filas · {panel['product_id'].n_unique()} productos")

# ── Totales para los shares (todos del mes t: contexto, no futuro) ──────
for niv in ("cat1", "cat2", "cat3"):
    t = panel.group_by([niv, "m"]).agg(pl.col("tn").sum().alias(f"tn_{niv}"))
    panel = panel.join(t, on=[niv, "m"], how="left")
mercado = panel.group_by("m").agg(pl.col("tn").sum().alias("tn_mercado"))
panel = panel.join(mercado, on="m", how="left")


def div_segura(num, den, nombre):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then(pl.col(num) / pl.col(den)).otherwise(0.0).alias(nombre))


SHARES = [f"sh_{n}" for n in ("cat1", "cat2", "cat3", "mercado")]
panel = panel.with_columns([div_segura("tn", f"tn_{n}", f"sh_{n}")
                            for n in ("cat1", "cat2", "cat3", "mercado")])

# ── Lags, promedios moviles, deltas e indices ───────────────────────────
df = panel.sort(["product_id", "m"]).with_columns(
    *[pl.col("tn").shift(k).over("product_id").alias(f"tn_lag{k}") for k in range(1, L + 1)],
    *[pl.col("tn").rolling_mean(w).over("product_id").alias(f"tn_ma{w}") for w in (3, 6, 12)],
    *[pl.col(s).shift(k).over("product_id").alias(f"{s}_lag{k}")
      for s in SHARES for k in (1, 2, 3)],
    *[pl.col(s).rolling_mean(3).over("product_id").alias(f"{s}_ma3") for s in SHARES],
    pl.col("n_clientes").shift(1).over("product_id").alias("n_clientes_lag1"),
    pl.col("n_clientes").rolling_mean(3).over("product_id").alias("n_clientes_ma3"),
    pl.col("req_qty").shift(1).over("product_id").alias("qty_lag1"),
    pl.col("tn").cum_max().over("product_id").alias("tn_pico_hasta_aca"),
    (pl.col("tn") > 0).cast(pl.Int8).alias("vendio"),
)

df = df.with_columns(
    *[(pl.col(s) - pl.col(f"{s}_lag1")).alias(f"{s}_d1") for s in SHARES],
    *[(pl.col(s) - pl.col(f"{s}_ma3")).alias(f"{s}_dma3") for s in SHARES],
    (pl.col("tn") - pl.col("tn_ma3")).alias("tn_dma3"),
    pl.col("vendio").rolling_mean(6).over("product_id").alias("frac_venta_6"),
    (pl.col("periodo") % 100).alias("mes_del_anio"),
    (pl.col("edad").is_between(0, 6)).cast(pl.Int8).alias("es_nuevo"),
)


def indice(num, den, nombre, techo=10.0):
    return (pl.when(pl.col(den).abs() > 1e-9)
              .then((pl.col(num) / pl.col(den)).clip(0.0, techo))
              .otherwise(pl.lit(None, dtype=pl.Float64)).alias(nombre))


df = df.with_columns(
    indice("tn", "tn_lag1", "idx_tn_mom"),
    indice("tn", "tn_ma3", "idx_tn_vs_ma3"),
    indice("tn", "tn_pico_hasta_aca", "idx_vs_pico"),
    indice("n_clientes", "n_clientes_lag1", "idx_clientes_mom"),
    indice("req_qty", "qty_lag1", "idx_qty_mom"),
)

# ── El target ───────────────────────────────────────────────────────────
df = df.sort(["product_id", "m"]).with_columns(
    pl.col("tn").shift(-H).over("product_id").alias("clase_tn"),
    ((((pl.col("m") + H - 1) // 12) * 100) + ((pl.col("m") + H - 1) % 12) + 1)
      .alias("periodo_objetivo"),
)

CATS = ["cat1", "cat2", "cat3", "brand"]
df = df.with_columns([pl.col(c).cast(pl.Utf8).fill_null("NA").cast(pl.Categorical)
                      for c in CATS])

NO_FEAT = {"product_id", "periodo", "m", "m_nace", "m_muere", "clase_tn",
           "periodo_objetivo"}
FEATURES = [c for c in df.columns if c not in NO_FEAT]

print(f"features: {len(FEATURES)}   filas: {df.height:,}   [{time.time()-t0:.0f}s]")
print(f"con target: {int(df['clase_tn'].is_not_null().sum()):,}")

## 3 — Partición y control de leakage

In [ ]:
sup = df.filter(pl.col("clase_tn").is_not_null())
periodos_sup = sorted(sup["periodo"].unique().to_list())
MESES_TRAIN = [m for m in PARAM['meses_train'] if m in periodos_sup]
MESES_VAL   = [m for m in PARAM['meses_val'] if m in periodos_sup]
MESES_TEST  = [m for m in PARAM['meses_test'] if m in periodos_sup]
MESES_INFER = sorted(df.filter(pl.col("clase_tn").is_null())["periodo"].unique().to_list())[-H:]
infer = df.filter(pl.col("periodo").is_in(MESES_INFER))

errores = []


def chk(ok, msg):
    print(f"  [{'ok   ' if ok else 'ERROR'}] {msg}")
    if not ok:
        errores.append(msg)


def a_m(p):
    return (p // 100) * 12 + (p % 100)


print("CONTROL DE LEAKAGE")
print("=" * 74)
for a, b, na, nb in ((MESES_TRAIN, MESES_VAL, "train", "val"),
                     (MESES_VAL, MESES_TEST, "val", "test")):
    g = a_m(min(b)) - a_m(max(a))
    chk(g >= H, f"gap {na}({max(a)}) -> {nb}({min(b)}) = {g} >= horizonte {H}")
if PARAM['reentrenar_con_val_para_test']:
    g = a_m(min(MESES_TEST)) - a_m(max(MESES_TRAIN + MESES_VAL))
    chk(g >= H, f"gap (train+val) -> test = {g} >= {H}")
chk(max(MESES_TRAIN) < min(MESES_VAL) < max(MESES_VAL) < min(MESES_TEST),
    "orden cronologico train < val < test")
chk("m_muere" not in FEATURES, "m_muere (dato del futuro) no es feature")
chk(not (set(FEATURES) & {"clase_tn", "periodo_objetivo"}), "el target no es feature")

# el shift del target, verificado fila por fila en la serie mas larga
_u = sup.group_by("product_id").agg(pl.len().alias("n")).sort("n", descending=True).head(1)
_s = df.filter(pl.col("product_id") == _u["product_id"][0]).sort("m")
_tn, _cl = _s["tn"].to_list(), _s["clase_tn"].to_list()
_mal = [i for i in range(len(_tn) - H)
        if _cl[i] is not None and abs(_cl[i] - _tn[i + H]) > 1e-9]
chk(not _mal, f"clase_tn[i] == tn[i+{H}] en el producto {_u['product_id'][0]} "
              f"({len(_tn)} meses, {len(_mal)} discrepancias)")

print("=" * 74)
if errores:
    raise RuntimeError(f"Leakage: {errores}")
print(f"TRAIN {len(MESES_TRAIN)} meses ({sup.filter(pl.col('periodo').is_in(MESES_TRAIN)).height:,} filas)"
      f" · VAL {MESES_VAL} · TEST {MESES_TEST} · INFER {MESES_INFER}")

## 4 — El WAPE y los baselines

El baseline es lo que se le regala al modelo: la estimación del **nivel**, que ya
funciona. El modelo después sólo tiene que aprender la **desviación**.

El más interesante es `lineal`: una Ridge sobre los lags, o sea **un promedio móvil con
los pesos aprendidos en vez de fijos**. Es la versión formal de la regresión stepwise que
dio 0,23, y por eso está acá como candidato de primera clase y no como curiosidad.

Ridge y no mínimos cuadrados puros porque los lags están muy correlacionados entre sí
(`tn_lag1` y `tn_lag2` se parecen mucho): sin regularizar, los coeficientes se vuelven
grandes y de signos alternados, y el modelo deja de generalizar.

In [ ]:
def wape(y_real, y_pred, ids=None) -> float:
    """WAPE en toneladas, agregando por producto. Identico al del pipe."""
    yr = np.asarray(y_real, dtype=np.float64)
    yp = np.maximum(np.asarray(y_pred, dtype=np.float64), 0.0)
    if ids is not None:
        _, inv = np.unique(np.asarray(ids), return_inverse=True)
        yr, yp = np.bincount(inv, weights=yr), np.bincount(inv, weights=yp)
    den = np.abs(yr).sum()
    return float("nan") if den == 0 else float(np.abs(yr - yp).sum() / den)


def bloque(meses):
    return sup.filter(pl.col("periodo").is_in(meses))


tr, va, te = bloque(MESES_TRAIN), bloque(MESES_VAL), bloque(MESES_TEST)
COLS_LIN = ["tn"] + [f"tn_lag{k}" for k in range(1, L + 1)] + ["tn_ma3", "tn_ma6"]


def X_lin(b):
    """Matriz para la parte lineal: los niveles recientes, con los nulos del arranque en 0."""
    return b.select(COLS_LIN).fill_null(0.0).to_numpy()


def wape_de(b, pred):
    return wape(b["clase_tn"].to_numpy(), pred, b["product_id"].to_numpy())


# ── Los baselines ────────────────────────────────────────────────────────
def baseline_fijo(b, cual):
    if cual == "tn0":
        return b["tn"].to_numpy().astype(np.float64)
    if cual == "ma_pond":
        return (0.5 * b["tn"].fill_null(0).to_numpy()
                + 0.3 * b["tn_lag1"].fill_null(0).to_numpy()
                + 0.2 * b["tn_lag2"].fill_null(0).to_numpy())
    return b[f"tn_{cual}"].fill_null(0.0).to_numpy().astype(np.float64)


_ridge_base = Ridge(alpha=PARAM['ridge_alpha'])
_ridge_base.fit(X_lin(tr), tr["clase_tn"].to_numpy())


def baseline_de(b, cual, ridge=None):
    """Prediccion del baseline para el bloque b. Siempre >= 0."""
    if cual == "lineal":
        r = ridge if ridge is not None else _ridge_base
        return np.maximum(r.predict(X_lin(b)), 0.0)
    return np.maximum(baseline_fijo(b, cual), 0.0)


CANDIDATOS = ["tn0", "ma3", "ma6", "ma12", "ma_pond", "lineal"]
print(f"{'baseline':10s} {'WAPE val':>10s}")
print("-" * 22)
wape_base = {}
for c in CANDIDATOS:
    wape_base[c] = wape_de(va, baseline_de(va, c))
    print(f"{c:10s} {wape_base[c]:10.5f}")

BASELINE = (PARAM['baseline'] if PARAM['baseline'] != 'auto'
            else min(wape_base, key=wape_base.get))
print(f"\nbaseline elegido: {BASELINE}"
      + ("  (por validacion)" if PARAM['baseline'] == 'auto' else "  (forzado)"))

# ── Los pesos que aprendio la Ridge: el promedio movil optimizado ────────
_co = dict(zip(COLS_LIN, _ridge_base.coef_))
print(f"\npesos de la Ridge (intercepto {_ridge_base.intercept_:+.3f}):")
for k, v in sorted(_co.items(), key=lambda kv: -abs(kv[1]))[:8]:
    print(f"   {k:10s} {v:+.4f}")
print("Si los pesos son positivos y decrecientes, la Ridge encontro sola un promedio")
print("movil ponderado. Si alternan de signo, esta capturando reversion a la media.")

## 5 — Los seis esquemas

Todos con las mismas features, la misma partición y la misma semilla. La única
diferencia es **quién se queda con qué parte del problema**.

El WAPE se mide siempre sobre **toneladas reconstruidas**, así que los seis números son
directamente comparables entre sí y con los del pipe.

In [ ]:
PARAMS_LGBM = dict(objective="regression", metric="mae", verbosity=-1,
                   n_estimators=500, learning_rate=0.05, num_leaves=63,
                   min_child_samples=20, subsample=0.9, subsample_freq=1,
                   colsample_bytree=0.8, seed=PARAM['semilla'], n_jobs=-1,
                   deterministic=True, force_row_wise=True)


def fit_lgbm(b, target, params=None, lineal=False):
    p = dict(params or PARAMS_LGBM)
    if lineal:
        # linear_tree ajusta una REGRESION LINEAL dentro de cada hoja: le da al arbol
        # la capacidad de extrapolar que por construccion no tiene.
        p.update(linear_tree=True, linear_lambda=1.0)
    m = lgb.LGBMRegressor(**p)
    bp = b.to_pandas()
    m.fit(bp[FEATURES], bp[target].to_numpy() if hasattr(bp[target], "to_numpy") else bp[target],
          categorical_feature=CATS)
    return m


def evaluar_esquemas(meses_fit, b_eval):
    """Devuelve {esquema: pred_tn} para el bloque de evaluacion."""
    fit = bloque(meses_fit)
    base_fit = baseline_de(fit, BASELINE)
    base_ev = baseline_de(b_eval, BASELINE)
    ev = b_eval.to_pandas()
    out, modelos = {}, {}

    # A) el baseline solo
    out["A_baseline"] = base_ev

    # B) LightGBM al nivel
    mB = fit_lgbm(fit, "clase_tn")
    out["B_lgbm_nivel"] = mB.predict(ev[FEATURES]); modelos["B_lgbm_nivel"] = mB

    # C) lineal al nivel (Ridge sobre TODAS las features numericas, no solo los lags)
    _num = [c for c in FEATURES if c not in CATS]
    mC = Ridge(alpha=PARAM['ridge_alpha'])
    mC.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
    out["C_lineal_nivel"] = mC.predict(b_eval.select(_num).fill_null(0.0).to_numpy())
    modelos["C_lineal_nivel"] = mC

    # D) LightGBM al residuo del baseline
    fit_d = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_fit))
    mD = fit_lgbm(fit_d, "y_res")
    out["D_lgbm_residuo"] = base_ev + mD.predict(ev[FEATURES]); modelos["D_lgbm_residuo"] = mD

    # E) lineal para el nivel + LightGBM para lo que sobra
    #    Es la apuesta: la recta hace el nivel (extrapola), el arbol las interacciones.
    rE = Ridge(alpha=PARAM['ridge_alpha'])
    rE.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    base_lin_fit = np.maximum(rE.predict(X_lin(fit)), 0.0)
    base_lin_ev = np.maximum(rE.predict(X_lin(b_eval)), 0.0)
    fit_e = fit.with_columns(
        pl.Series("y_res", fit["clase_tn"].to_numpy() - base_lin_fit))
    mE = fit_lgbm(fit_e, "y_res")
    out["E_lineal_mas_lgbm"] = base_lin_ev + mE.predict(ev[FEATURES])
    modelos["E_lineal_mas_lgbm"] = (rE, mE)

    # F) LightGBM con hojas lineales, al nivel
    mF = fit_lgbm(fit, "clase_tn", lineal=True)
    out["F_lgbm_hojas_lineales"] = mF.predict(ev[FEATURES]); modelos["F_lgbm_hojas_lineales"] = mF

    return out, modelos


t0 = time.time()
pred_val, mod_val = evaluar_esquemas(MESES_TRAIN, va)
print(f"[{time.time()-t0:.0f}s]\n")

ESQUEMAS = list(pred_val)
print(f"{'esquema':24s} {'WAPE val':>10s}")
print("-" * 36)
wape_val = {}
for e in ESQUEMAS:
    wape_val[e] = wape_de(va, pred_val[e])
    print(f"{e:24s} {wape_val[e]:10.5f}")

ESQUEMA = (PARAM['esquema'] if PARAM['esquema'] != 'auto'
           else min(wape_val, key=wape_val.get))
print(f"\nesquema elegido: {ESQUEMA}"
      + ("  (por validacion)" if PARAM['esquema'] == 'auto' else "  (forzado)"))
_mej = 100 * (wape_val['A_baseline'] - wape_val[ESQUEMA]) / wape_val['A_baseline']
print(f"mejora sobre el baseline solo: {_mej:+.1f}%")

## 6 — Optuna sobre el esquema ganador

Sólo se afina el esquema que ganó en validación, y **lo que se minimiza es el WAPE en
toneladas reconstruidas**, nunca el error del residuo. Si el esquema ganador es
`A_baseline` no hay nada que optimizar y esta celda lo saltea — que sería, en sí, el
resultado más interesante posible: que ningún modelo le gana al promedio.

In [ ]:
def espacio(trial):
    return dict(
        objective="regression", metric="mae", verbosity=-1,
        seed=PARAM['semilla'], n_jobs=-1, subsample_freq=1,
        deterministic=True, force_row_wise=True,
        num_leaves=trial.suggest_int("num_leaves", 15, 255),
        max_depth=trial.suggest_int("max_depth", 3, 12),
        learning_rate=trial.suggest_float("learning_rate", 5e-3, 0.2, log=True),
        n_estimators=trial.suggest_int("n_estimators", 200, PARAM['techo_arboles']),
        min_child_samples=trial.suggest_int("min_child_samples", 5, 200),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    )


def predecir_esquema(esquema, meses_fit, b_eval, params, semilla=None):
    """Entrena el esquema con `params` y devuelve las toneladas predichas."""
    fit = bloque(meses_fit)
    ev = b_eval.to_pandas()
    p = dict(params)
    if semilla is not None:
        p["seed"] = semilla

    if esquema == "A_baseline":
        return baseline_de(b_eval, BASELINE), None

    if esquema == "C_lineal_nivel":
        _num = [c for c in FEATURES if c not in CATS]
        r = Ridge(alpha=PARAM['ridge_alpha'])
        r.fit(fit.select(_num).fill_null(0.0).to_numpy(), fit["clase_tn"].to_numpy())
        return r.predict(b_eval.select(_num).fill_null(0.0).to_numpy()), r

    if esquema == "B_lgbm_nivel":
        m = fit_lgbm(fit, "clase_tn", p)
        return m.predict(ev[FEATURES]), m

    if esquema == "F_lgbm_hojas_lineales":
        m = fit_lgbm(fit, "clase_tn", p, lineal=True)
        return m.predict(ev[FEATURES]), m

    if esquema == "D_lgbm_residuo":
        bf, be = baseline_de(fit, BASELINE), baseline_de(b_eval, BASELINE)
        f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
        m = fit_lgbm(f2, "y_res", p)
        return be + m.predict(ev[FEATURES]), m

    # E_lineal_mas_lgbm
    r = Ridge(alpha=PARAM['ridge_alpha'])
    r.fit(X_lin(fit), fit["clase_tn"].to_numpy())
    bf = np.maximum(r.predict(X_lin(fit)), 0.0)
    be = np.maximum(r.predict(X_lin(b_eval)), 0.0)
    f2 = fit.with_columns(pl.Series("y_res", fit["clase_tn"].to_numpy() - bf))
    m = fit_lgbm(f2, "y_res", p)
    return be + m.predict(ev[FEATURES]), (r, m)


SIN_HIPER = {"A_baseline", "C_lineal_nivel"}

if ESQUEMA in SIN_HIPER:
    print(f"El esquema ganador ({ESQUEMA}) no tiene hiperparametros que buscar.")
    if ESQUEMA == "A_baseline":
        print("Y eso es un resultado en si mismo: ningun modelo le gana al baseline.")
    study = None
    MEJORES = {}
else:
    study = optuna.create_study(
        direction="minimize", study_name=EXPERIMENTO,
        sampler=optuna.samplers.TPESampler(seed=PARAM['semilla']),
        storage=f"sqlite:///{Path.home() / ('optuna_' + EXPERIMENTO + '.db')}",
        load_if_exists=True)

    def objective(trial):
        pred, _ = predecir_esquema(ESQUEMA, MESES_TRAIN, va, espacio(trial))
        return wape_de(va, pred)

    t0 = time.time()
    study.optimize(objective, n_trials=PARAM['n_trials'])
    MEJORES = {**espacio(optuna.trial.FixedTrial(study.best_params))}
    print(f"{len(study.trials)} trials · mejor WAPE val = {study.best_value:.5f}"
          f"  ({time.time()-t0:.0f}s)")
    print(f"mejora de Optuna sobre los defaults: "
          f"{100*(wape_val[ESQUEMA]-study.best_value)/wape_val[ESQUEMA]:+.1f}%")
    for k, v in study.best_params.items():
        print(f"   {k:22s} {v}")

## 7 — Resultados: la tabla completa

`val` es optimista para el esquema ganador (Optuna lo minimizó). `test` es el número
honesto, medido una sola vez.

In [ ]:
MESES_FIT_TEST = (MESES_TRAIN + MESES_VAL) if PARAM['reentrenar_con_val_para_test'] else MESES_TRAIN
_pars = MEJORES or PARAMS_LGBM

pred_test, _ = evaluar_esquemas(MESES_FIT_TEST, te)
if MEJORES:
    pred_test[ESQUEMA], _ = predecir_esquema(ESQUEMA, MESES_FIT_TEST, te, _pars)

print(f"{'esquema':24s} {'WAPE val':>10s} {'WAPE test':>10s}")
print("-" * 48)
METRICAS = {}
for e in ESQUEMAS:
    wt = wape_de(te, pred_test[e])
    METRICAS[e] = {"val": wape_val[e], "test": wt}
    marca = "  <-" if e == ESQUEMA else ""
    print(f"{e:24s} {wape_val[e]:10.5f} {wt:10.5f}{marca}")

_a, _g = METRICAS['A_baseline']['test'], METRICAS[ESQUEMA]['test']
print(f"\nganador en test: {min(METRICAS, key=lambda e: METRICAS[e]['test'])}")
print(f"{ESQUEMA} vs baseline solo, en test: {100*(_a-_g)/_a:+.1f}%")
_br = METRICAS[ESQUEMA]['test'] - METRICAS[ESQUEMA]['val']
print(f"brecha test - val: {_br:+.5f}"
      + ("   <- sobreajuste a validacion" if _br > 0.02 else ""))

pl.DataFrame([{"esquema": e, **METRICAS[e]} for e in ESQUEMAS]) \
  .write_csv(DIR_OUT / "esquemas.csv")

# ── Donde gana cada uno: por volumen del producto ───────────────────────
# El WAPE pondera por volumen, asi que lo unico que mueve la aguja son los productos
# grandes. Este corte dice si la ventaja viene de ahi o de la cola.
det = te.select("product_id", "clase_tn").with_columns(
    pl.Series("base", pred_test["A_baseline"]),
    pl.Series("gana", pred_test[ESQUEMA]))
_q = det.group_by("product_id").agg(pl.col("clase_tn").sum().alias("tn"))
_cortes = _q["tn"].qcut(4, labels=["Q1 chico", "Q2", "Q3", "Q4 grande"], allow_duplicates=True)
_q = _q.with_columns(_cortes.alias("cuartil"))
det = det.join(_q.select("product_id", "cuartil"), on="product_id", how="left")

filas = []
for c in ["Q1 chico", "Q2", "Q3", "Q4 grande"]:
    b = det.filter(pl.col("cuartil") == c)
    if b.height < 5:
        continue
    filas.append({"cuartil": c, "n_filas": b.height,
                  "tn_real": round(float(b["clase_tn"].sum()), 1),
                  "wape_baseline": round(wape(b["clase_tn"], b["base"], b["product_id"]), 4),
                  f"wape_{ESQUEMA}": round(wape(b["clase_tn"], b["gana"], b["product_id"]), 4)})
por_vol = pl.DataFrame(filas)
print()
print(por_vol)
por_vol.write_csv(DIR_OUT / "por_cuartil_de_volumen.csv")
print("\nEl WAPE pondera por volumen: si la ventaja no esta en Q4, no va a mover el")
print("numero global aunque se vea grande en los cuartiles chicos.")

## 8 — Entrenamiento final y entrega

Se reentrena el esquema ganador con **todos** los meses supervisados y se predice el mes
objetivo. Del modelo final no hay métrica honesta: la que se reporta es la de `test`.

In [ ]:
MESES_TODOS = sorted(periodos_sup)
print(f"reentrenando {ESQUEMA} con {len(MESES_TODOS)} meses "
      f"({MESES_TODOS[0]}..{MESES_TODOS[-1]})")

preds = []
for sem in PARAM['semillas_ensemble']:
    p, _ = predecir_esquema(ESQUEMA, MESES_TODOS, infer, _pars, semilla=sem)
    preds.append(p)
    print(f"  semilla {sem} lista")
pred_infer_tn = np.maximum(np.mean(preds, axis=0), PARAM['clip_min'])

pred_infer = infer.select("product_id", "periodo", "periodo_objetivo").with_columns(
    pl.Series("tn_pred", pred_infer_tn),
    pl.Series("baseline", baseline_de(infer, BASELINE)))
pred_infer.write_parquet(DIR_OUT / "predicciones_inferencia.parquet")

print(f"\npredicciones: {pred_infer.height:,} filas")
print(pred_infer.group_by("periodo", "periodo_objetivo").len().sort("periodo"))
print(f"\ntn_pred   min {pred_infer_tn.min():.2f}   media {pred_infer_tn.mean():.2f}   "
      f"max {pred_infer_tn.max():.2f}")
print(f"correlacion con el baseline: "
      f"{np.corrcoef(pred_infer_tn, pred_infer['baseline'].to_numpy())[0,1]:.4f}")
print("Si esa correlacion es ~1, el modelo esta repitiendo el baseline y no aporta nada.")

## 9 — El CSV y el submit

In [ ]:
OBJ = PARAM['periodo_objetivo']
obj = pred_infer.filter(pl.col("periodo_objetivo") == OBJ)
if obj.is_empty():
    raise RuntimeError(f"No hay predicciones para {OBJ}. Disponibles: "
                       f"{sorted(pred_infer['periodo_objetivo'].unique().to_list())}")

por_producto = obj.group_by("product_id").agg(pl.col("tn_pred").sum().alias("tn"))
oficiales = pl.read_csv(DIR_RAW / "product_id_apredecir201912.txt")
submit = oficiales.select("product_id").join(por_producto, on="product_id", how="left")
sin_pred = int(submit["tn"].null_count())
submit = submit.with_columns(pl.col("tn").fill_null(0.0)).sort("product_id")

print(f"mes objetivo {OBJ}: {obj.height} filas -> {por_producto.height} productos")
print(f"lista oficial: {oficiales.height}   sin prediccion (van en 0): {sin_pred}")
if sin_pred > oficiales.height * 0.05:
    print("   ATENCION: mas del 5% de la lista. Revisalo antes de subir.")
print(f"\ntn   min {submit['tn'].min():.3f}   media {submit['tn'].mean():.3f}   "
      f"max {submit['tn'].max():.3f}   suma {submit['tn'].sum():,.1f}")
print(submit.head(10))

path_submit = DIR_OUT / f"submission_{OBJ}.csv"
submit.write_csv(path_submit)
shutil.copy(path_submit, RUTA_EXP / "submission_ultima.csv")
print(f"\nGuardado: {path_submit}")


def kaggle_cli(args):
    try:
        r = subprocess.run(["kaggle"] + args, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout or "") + (r.stderr or "")
    except FileNotFoundError:
        return False, "La CLI de kaggle no esta instalada.  pip install kaggle"
    except Exception as e:
        return False, f"{type(e).__name__}: {e}"


if not PARAM['submit']:
    print("\nPARAM['submit'] = False -> no se sube. El CSV ya esta generado.")
else:
    kd = Path.home() / ".kaggle" / "kaggle.json"
    kd.parent.mkdir(parents=True, exist_ok=True)
    if not kd.exists():
        for cand in (BUCKET / "kaggle.json", BUCKET / "kaggle" / "kaggle.json"):
            if cand.exists():
                shutil.copy(cand, kd); kd.chmod(0o600); break
    if not kd.exists():
        print("\nSin credenciales de Kaggle. El CSV ya esta generado.")
    else:
        kd.chmod(0o600)
        msg = PARAM['mensaje_submit'] or (
            f"{ESQUEMA} sobre {BASELINE} | wape_test={METRICAS[ESQUEMA]['test']:.5f}")
        ok, salida = kaggle_cli(["competitions", "submit",
                                 "-c", PARAM['kaggle_competition'],
                                 "-f", str(path_submit), "-m", msg])
        print(f"\nmensaje: {msg}\n{salida}")
        print("Submit enviado." if ok else "NO se pudo subir; el CSV esta en disco.")

## 10 — Registro y leaderboard

Una fila por experimento en `exp_residuo/leaderboard_residuo.csv`, con **el WAPE de los
seis esquemas en cada corrida**. Así se ve si el ranking entre esquemas es estable al
cambiar el baseline o las features, que es lo que decide si el hallazgo es real.

In [ ]:
resultado = {
    'experimento': EXPERIMENTO,
    'idea': 'residuo sobre baseline: el nivel lo hace el baseline, la desviacion el modelo',
    'granularidad': 'producto-mes',
    'baseline_elegido': BASELINE, 'wape_baselines_val': wape_base,
    'esquema_elegido': ESQUEMA, 'metricas_por_esquema': METRICAS,
    'horizonte': H, 'max_lags': L,
    'meses_train': MESES_TRAIN, 'meses_val': MESES_VAL, 'meses_test': MESES_TEST,
    'meses_inferencia': MESES_INFER, 'periodo_objetivo': OBJ,
    'n_features': len(FEATURES), 'features': FEATURES,
    'n_trials': len(study.trials) if study else 0,
    'hiperparametros': study.best_params if study else {},
    'pesos_ridge_baseline': {k: round(float(v), 5) for k, v in _co.items()},
    'ridge_alpha': PARAM['ridge_alpha'],
    'n_sin_prediccion': sin_pred,
    'tn_total': float(submit['tn'].sum()),
    'semilla': PARAM['semilla'],
}
with open(DIR_OUT / "resultado.json", "w", encoding="utf-8") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False, default=str)

fila = {'experimento': EXPERIMENTO, 'baseline': BASELINE, 'esquema': ESQUEMA,
        'max_lags': L, 'n_features': len(FEATURES),
        'wape_test_ganador': round(METRICAS[ESQUEMA]['test'], 5),
        'wape_val_ganador': round(METRICAS[ESQUEMA]['val'], 5),
        **{f"test_{e}": round(METRICAS[e]['test'], 5) for e in ESQUEMAS},
        'mejora_vs_baseline_pct': round(100*(_a-_g)/_a, 2),
        'sin_prediccion': sin_pred, 'tn_total': round(float(submit['tn'].sum()), 1)}

path_lb = RUTA_EXP / "leaderboard_residuo.csv"
nueva = pl.DataFrame([fila])
if path_lb.exists():
    viejo = pl.read_csv(path_lb).filter(pl.col("experimento") != EXPERIMENTO)
    nueva = pl.concat([viejo, nueva], how="diagonal_relaxed")
nueva.sort("wape_test_ganador").write_csv(path_lb)

print(f"Archivos en {DIR_OUT.relative_to(BUCKET)}:")
for p in sorted(DIR_OUT.iterdir()):
    print(f"  - {p.name}")
print(f"\nleaderboard_residuo.csv ({nueva.height} experimentos):")
print(nueva.select("baseline", "esquema", "wape_test_ganador",
                   "test_A_baseline", "mejora_vs_baseline_pct"))

## Cómo leer el resultado

La tabla de la sección 7 contesta la pregunta directamente. Cuatro desenlaces posibles,
y ninguno es un fracaso:

| Si gana… | Significa |
|---|---|
| **A_baseline** | Ningún modelo le gana a un promedio. Es un resultado fuerte y publicable: en esta serie el ruido domina y la complejidad no compra nada. |
| **C_lineal_nivel** | Se confirma tu observación: la señal es lineal en los niveles recientes, y el GBM está de más. |
| **E_lineal_mas_lgbm** | La hipótesis del notebook: la recta hace el nivel y el árbol agrega algo sobre el residuo. El mejor de los dos mundos. |
| **B** o **F** | El GBM sí puede con el nivel, y lo de la regresión lineal era un problema de tuneo, no estructural. |

Y hay un número que conviene mirar antes de festejar: **la correlación entre la
predicción final y el baseline**, que imprime la sección 8. Si es 0,99, el modelo está
repitiendo el promedio con pasos extra — el WAPE puede mejorar un poco y aun así no
haber aprendido nada.

### Qué variar después

1. **El baseline.** `ma3` contra `lineal` es la comparación que más importa, porque
   `lineal` *es* el promedio con pesos aprendidos.
2. **`max_lags`.** Si la Ridge le pone peso a `tn_lag11` y `tn_lag12`, hay estacionalidad
   anual y conviene subirlo.
3. **`ridge_alpha`.** Con lags muy correlacionados, este número decide si los
   coeficientes salen estables o alternando de signo. Mirá los pesos que imprime la
   sección 4.